<div dir="rtl">
<h1>حافظهٔ یک‌نویسه‌ای را آزمایش کنید</h1>
<p>درس 4 از 76 · نسخهٔ صفر: آیا شمارش می‌تواند متن بسازد؟ · <code dir="ltr">03-counts</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-01/chapter-03/03-counts.html">📖 بازگشت به همین درس</a></p>
<p>جدول انتقال را خودتان بسازید و با نسخهٔ واقعی شمارشی مقایسه کنید.</p><p>پیش‌نیاز: درس 02-token و دیکشنری؛ Sampling و Seed را در درس جاری بخوانید.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>آیا دو متن آغازین متفاوت که هر دو به «م» ختم می‌شوند، با Seed یکسان ادامه‌های متفاوتی می‌گیرند؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
from collections import Counter, defaultdict
from mini_gpt.stages.v0 import transition_counts, generate
ids = [1, 0, 1, 0]
print("Input IDs:", ids)

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع count_pairs(ids) دیکشنری current → دیکشنری next → count برگرداند. فقط همسایه‌های واقعی دنباله را بشمارید؛ میان آخر و اول پیوند تازه نسازید.</p>
</div>

In [ ]:
def count_pairs(ids):
    # TODO: count adjacent pairs
    return None

In [ ]:
def test_exercise():
    result = count_pairs(ids)
    if result is None:
        return False
    assert result == {1: {0: 2}, 0: {1: 1}}
    assert count_pairs([7, 7, 7]) == {7: {7: 2}}
    assert count_pairs([1]) == {}
    assert result == transition_counts(ids)
    return True

exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>فقط ابتدای Prompt را عوض کنید و آخرین ID را ثابت نگه دارید. نسخهٔ واقعی v0 را با Seed ثابت صدا بزنید؛ فقط بخش تازهٔ خروجی را مقایسه کنید.</p>
</div>

In [ ]:
counts = transition_counts(ids)
first = generate(counts, [0, 1], 12, 2, seed=42)
second = generate(counts, [1, 1, 1], 12, 2, seed=42)
print(first[2:], second[3:])
assert first[2:] == second[3:]

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>جمع‌زدن شمارش تمام سطرها، Context را از بین می‌برد. تابع next_weights فقط سطر آخرین ID را بخواند و برای هر ID مجاز یک واحد هموارسازی اضافه کند.</p>
</div>

In [ ]:
rows = {0: {0: 8}, 1: {1: 8}}
wrong = [sum(row.get(i, 0) for row in rows.values())+1 for i in range(2)]
print("Broken weights for either context:", wrong)
assert wrong == [9, 9]

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def next_weights(counts, current, vocabulary_size):
    # TODO: use only the current row, with add-one smoothing
    return None

In [ ]:
def test_repair():
    result = next_weights(rows, 0, 2)
    if result is None:
        return False
    assert result == [9, 1]
    assert next_weights(rows, 1, 2) == [1, 9]
    assert next_weights(rows, 9, 2) == [1, 1]
    return True

repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: implement the TODO and rerun')

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>count_pairs را مستقیماً با transition_counts واقعی مقایسه کردیم و برای تولید از generate در mini_gpt/stages/v0.py استفاده کردیم؛ هیچ شبکه‌ای در این آزمایش نیست.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>با کدام شاهد نشان دادید محدودیت مدل در اطلاعات ورودی آن است، نه در Seed نامناسب؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-01/chapter-03/03-counts.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/03-counts.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>